# 02. 베이스라인 비교

**목적**: PPO가 이겨야 할 비교 기준(Buy-and-Hold, Fixed Grid, ATR Grid)의 성능을 Val 셋에서 시각화한다.

**평가 지표**: Sharpe Ratio (주), 누적 수익률, MDD, 거래횟수, 완료사이클

**데이터**: `data/processed/btc_val.parquet` (2023년, 강세장)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

from src.utils.config import load_config
from src.agents.baselines import run_all_baselines
from src.evaluation.metrics import compute_all

import os
os.makedirs('../reports/semester1/figures', exist_ok=True)

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [2]:
config = load_config('../config/experiment_config.yaml')
df_val = pd.read_parquet('../data/processed/btc_val.parquet')
initial_cash = config['environment']['initial_cash']

print(f'Val 데이터: {len(df_val):,}행  ({df_val.index[0]} ~ {df_val.index[-1]})')
print(f'초기 자본: ${initial_cash:,.0f}')

Val 데이터: 8,736행  (2023-01-01 00:00:00+00:00 ~ 2023-12-31 00:00:00+00:00)
초기 자본: $10,000


## 1. 전체 베이스라인 실행

In [3]:
results = run_all_baselines(df_val, config)

# 지표 계산
metrics = {}
for name, r in results.items():
    m = compute_all(r['equity_curve'], initial_cash, r['n_trades'], r['completed_cycles'])
    metrics[name] = m

metrics_df = pd.DataFrame(metrics).T
metrics_df = metrics_df[['total_return_pct', 'sharpe_ratio', 'max_drawdown_pct',
                          'n_trades', 'n_cycles', 'avg_cycle_pnl_pct', 'avg_cycle_hours']]
metrics_df.columns = ['수익률(%)', 'Sharpe', 'MDD(%)', '거래수', '사이클수', '사이클수익(%)', '사이클시간(봉)']
print(metrics_df.round(3).to_string())

                  수익률(%)  Sharpe  MDD(%)     거래수   사이클수  사이클수익(%)  사이클시간(봉)
buy_and_hold     150.177   2.377  21.741     1.0    0.0     0.000     0.000
fixed_grid_1pct   43.164   2.610  10.769   567.0  141.0     0.255    45.071
fixed_grid_2pct   16.963   2.032   7.652   126.0   41.0     0.382    86.512
fixed_grid_5pct    2.470   1.375   1.665     8.0    4.0     0.612   157.000
atr_grid_k0.5     24.701   1.118  16.068  1464.0  347.0     0.067    23.507
atr_grid_k1.0     39.794   1.434  15.864   833.0  213.0     0.158    34.657
atr_grid_k2.0     28.980   1.948   9.376   320.0   91.0     0.280    59.385


## 2. 성능 지표 비교표 (Bar Chart)

In [4]:
labels = list(metrics.keys())
sharpes = [metrics[k]['sharpe_ratio'] for k in labels]
returns = [metrics[k]['total_return_pct'] for k in labels]
mdds    = [metrics[k]['max_drawdown_pct'] for k in labels]

# 색상: buy_and_hold=파랑, fixed_grid=초록, atr_grid=주황
colors = []
for k in labels:
    if 'buy_and_hold' in k:  colors.append('#2196F3')
    elif 'fixed' in k:       colors.append('#4CAF50')
    else:                    colors.append('#FF9800')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('베이스라인 성능 비교 (Val 셋, 2023)', fontsize=14, fontweight='bold')

label_names = [
    'Buy&Hold',
    'Fixed\n1%', 'Fixed\n2%', 'Fixed\n5%',
    'ATR\nk=0.5', 'ATR\nk=1.0', 'ATR\nk=2.0'
]
x = range(len(labels))

# Sharpe
axes[0].bar(x, sharpes, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Sharpe Ratio (연율화)\n↑ 높을수록 좋음')
axes[0].set_xticks(x); axes[0].set_xticklabels(label_names, fontsize=8)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
for i, v in enumerate(sharpes):
    axes[0].text(i, v + 0.03, f'{v:.2f}', ha='center', va='bottom', fontsize=8)

# 수익률
axes[1].bar(x, returns, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('누적 수익률 (%)\n↑ 높을수록 좋음')
axes[1].set_xticks(x); axes[1].set_xticklabels(label_names, fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
for i, v in enumerate(returns):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

# MDD
axes[2].bar(x, mdds, color=colors, edgecolor='white', linewidth=0.5)
axes[2].set_title('최대 낙폭 MDD (%)\n↓ 낮을수록 좋음')
axes[2].set_xticks(x); axes[2].set_xticklabels(label_names, fontsize=8)
for i, v in enumerate(mdds):
    axes[2].text(i, v + 0.2, f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Buy-and-Hold'),
    Patch(facecolor='#4CAF50', label='Fixed Grid'),
    Patch(facecolor='#FF9800', label='ATR Grid'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('../reports/semester1/figures/02_baseline_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 02_baseline_metrics.png')

저장: 02_baseline_metrics.png


C:\Users\user\AppData\Local\Temp\ipykernel_55020\3024586402.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 자본 곡선 (Equity Curve)

In [5]:
fig, ax = plt.subplots(figsize=(14, 6))

style_map = {
    'buy_and_hold':   {'color': '#2196F3', 'lw': 2.5, 'ls': '-',  'label': 'Buy-and-Hold'},
    'fixed_grid_1pct':{'color': '#4CAF50', 'lw': 1.5, 'ls': '-',  'label': 'Fixed Grid 1%'},
    'fixed_grid_2pct':{'color': '#81C784', 'lw': 1.5, 'ls': '--', 'label': 'Fixed Grid 2%'},
    'fixed_grid_5pct':{'color': '#C8E6C9', 'lw': 1.5, 'ls': ':',  'label': 'Fixed Grid 5%'},
    'atr_grid_k0.5':  {'color': '#FF9800', 'lw': 1.5, 'ls': '-',  'label': 'ATR Grid k=0.5'},
    'atr_grid_k1.0':  {'color': '#FFB74D', 'lw': 1.5, 'ls': '--', 'label': 'ATR Grid k=1.0'},
    'atr_grid_k2.0':  {'color': '#FFE0B2', 'lw': 1.5, 'ls': ':',  'label': 'ATR Grid k=2.0'},
}

for name, r in results.items():
    eq = r['equity_curve']
    s  = style_map.get(name, {})
    ax.plot(eq.values, label=s.get('label', name),
            color=s.get('color', 'gray'),
            lw=s.get('lw', 1),
            ls=s.get('ls', '-'))

ax.axhline(initial_cash, color='black', lw=0.8, ls='--', alpha=0.5, label='초기 자본 ($10,000)')
ax.set_title('자본 곡선 비교 (Val 셋, 2023)', fontsize=13, fontweight='bold')
ax.set_xlabel('스텝 (시간봉)')
ax.set_ylabel('포트폴리오 가치 ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/02_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 02_equity_curves.png')

저장: 02_equity_curves.png


C:\Users\user\AppData\Local\Temp\ipykernel_55020\2250748437.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. 낙폭 (Drawdown) 시계열

In [6]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 상단: BTC 가격
axes[0].plot(df_val['close'].values, color='#607D8B', lw=1)
axes[0].set_ylabel('BTC 가격 ($)')
axes[0].set_title('BTC 가격 & 낙폭 비교 (Val 셋, 2023)', fontsize=12, fontweight='bold')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].grid(True, alpha=0.3)

# 하단: 낙폭
key_strategies = ['buy_and_hold', 'fixed_grid_1pct', 'atr_grid_k1.0']
dd_colors = {'buy_and_hold': '#2196F3', 'fixed_grid_1pct': '#4CAF50', 'atr_grid_k1.0': '#FF9800'}
dd_labels = {'buy_and_hold': 'Buy-and-Hold', 'fixed_grid_1pct': 'Fixed Grid 1%', 'atr_grid_k1.0': 'ATR Grid k=1.0'}

for name in key_strategies:
    eq = results[name]['equity_curve']
    rolling_peak = eq.cummax()
    dd = (rolling_peak - eq) / rolling_peak * 100
    axes[1].plot(dd.values, label=dd_labels[name], color=dd_colors[name], lw=1.2)

axes[1].set_ylabel('낙폭 (%)')
axes[1].set_xlabel('스텝 (시간봉)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../reports/semester1/figures/02_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 02_drawdown.png')

저장: 02_drawdown.png


C:\Users\user\AppData\Local\Temp\ipykernel_55020\1466836802.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. 사이클 분석 (그리드 전략)

In [7]:
grid_strategies = [k for k in results if k != 'buy_and_hold']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('그리드 전략 사이클 분석 (Val 셋, 2023)', fontsize=13, fontweight='bold')

# 사이클 수익률 분포
ax = axes[0]
data_pnl = []
labels_pnl = []
for name in grid_strategies:
    cycles = results[name]['completed_cycles']
    if cycles:
        pnls = [c['pnl_pct'] * 100 for c in cycles]
        data_pnl.append(pnls)
        short = name.replace('fixed_grid_', 'FG ').replace('atr_grid_', 'ATR ')
        labels_pnl.append(short)

if data_pnl:
    bp = ax.boxplot(data_pnl, tick_labels=labels_pnl, patch_artist=True, showfliers=False)
    colors_box = ['#4CAF50','#81C784','#C8E6C9','#FF9800','#FFB74D','#FFE0B2']
    for patch, c in zip(bp['boxes'], colors_box):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    ax.axhline(0, color='red', lw=0.8, ls='--', alpha=0.7)
    ax.set_title('사이클 수익률 분포 (%)')
    ax.set_ylabel('사이클 수익률 (%)')
    ax.tick_params(axis='x', rotation=30)

# 사이클 소요 시간
ax = axes[1]
data_hrs = []
for name in grid_strategies:
    cycles = results[name]['completed_cycles']
    if cycles:
        hrs = [c['cycle_hours'] for c in cycles]
        data_hrs.append(hrs)

if data_hrs:
    bp = ax.boxplot(data_hrs, tick_labels=labels_pnl, patch_artist=True, showfliers=False)
    for patch, c in zip(bp['boxes'], colors_box):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    ax.set_title('사이클 소요 시간 분포 (봉)')
    ax.set_ylabel('소요 시간 (봉 = 시간)')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../reports/semester1/figures/02_cycle_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 02_cycle_analysis.png')

저장: 02_cycle_analysis.png


C:\Users\user\AppData\Local\Temp\ipykernel_55020\3386003490.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. 요약 및 PPO 목표치

In [8]:
best_sharpe_name = max(metrics, key=lambda k: metrics[k]['sharpe_ratio'])
best_sharpe_val  = metrics[best_sharpe_name]['sharpe_ratio']

print('=' * 55)
print('베이스라인 요약 (Val 셋, 2023)')
print('=' * 55)
print(f"{'전략':<22} {'Sharpe':>7} {'수익률(%)':>10} {'MDD(%)':>8}")
print('-' * 55)
for name, m in metrics.items():
    marker = ' ★' if name == best_sharpe_name else ''
    print(f"{name:<22} {m['sharpe_ratio']:>7.3f} {m['total_return_pct']:>10.2f} {m['max_drawdown_pct']:>8.2f}{marker}")

print('=' * 55)
print(f'\nPPO 목표 Sharpe : {best_sharpe_val:.3f} ({best_sharpe_name}) 초과')
print(f'(2023 강세장 — Train 셋 비교가 더 공정한 평가 기준)')

베이스라인 요약 (Val 셋, 2023)
전략                      Sharpe     수익률(%)   MDD(%)
-------------------------------------------------------
buy_and_hold             2.377     150.18    21.74
fixed_grid_1pct          2.610      43.16    10.77 ★
fixed_grid_2pct          2.032      16.96     7.65
fixed_grid_5pct          1.375       2.47     1.66
atr_grid_k0.5            1.118      24.70    16.07
atr_grid_k1.0            1.434      39.79    15.86
atr_grid_k2.0            1.948      28.98     9.38

PPO 목표 Sharpe : 2.610 (fixed_grid_1pct) 초과
(2023 강세장 — Train 셋 비교가 더 공정한 평가 기준)
